In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/data.csv")
print(df.shape)

(42372, 1036)


In [25]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Repeated customer IDs:", df["CONS_NO"].duplicated().sum())

df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Fully duplicate rows: 0
Repeated customer IDs: 0
Shape after removing duplicates: (42372, 1036)


In [27]:
ids = df["CONS_NO"]
labels = df["FLAG"]
usage = df.drop(columns=["CONS_NO", "FLAG"])

# convert column names like "1/1/2014" into real dates, then sort them
usage.columns = pd.to_datetime(usage.columns, format="%m/%d/%Y")
usage = usage.sort_index(axis=1)

print("From:", usage.columns.min().date(), "To:", usage.columns.max().date())
print("Usage table shape:", usage.shape)

From: 2014-01-01 To: 2016-10-31
Usage table shape: (42372, 1034)


In [28]:
missing_per_customer = usage.isnull().mean(axis=1) * 100
print(missing_per_customer.describe())
print("Customers with more than 50% missing:", (missing_per_customer > 50).sum())
print("Negative values:", (usage < 0).sum().sum())

count    42372.000000
mean        25.639920
std         28.825828
min          0.096712
25%          1.257253
50%         10.638298
75%         62.016441
max        100.000000
dtype: float64
Customers with more than 50% missing: 11184
Negative values: 0


In [31]:
groups = pd.cut(
    missing_per_customer,
    bins=[-1, 10, 50, 90, 99.99, 100],
    labels=["0-10%", "10-50%", "50-90%", "90-99%", "100%"]
)

summary = pd.DataFrame({
    "customers": groups.value_counts().sort_index(),
    "theft_rate_%": (labels.groupby(groups).mean() * 100).round(2)
})
print(summary)

        customers  theft_rate_%
0-10%       20997          6.55
10-50%      10191         10.15
50-90%      11101         10.67
90-99%         78         23.08
100%            5         60.00


In [32]:
# gaps in the middle: connect the neighbouring days with a straight line
usage_filled = usage.interpolate(axis=1, limit_area="inside")

# gaps at the start/end: meter probably not active, so use 0
usage_filled = usage_filled.fillna(0)

print("Missing after filling:", usage_filled.isnull().sum().sum())

Missing after filling: 0


In [33]:
row_mean = usage_filled.mean(axis=1)
row_std = usage_filled.std(axis=1)
upper_limit = row_mean + 3 * row_std   # each customer gets their own limit

too_high = usage_filled.gt(upper_limit, axis=0).sum().sum()
print("Values capped:", too_high)

usage_clean = usage_filled.clip(lower=0, upper=upper_limit, axis=0)

Values capped: 555719


In [34]:
# keep customers that have at least 10% real readings
keep = missing_per_customer < 90
print("Customers removed:", (~keep).sum())

clean = usage_clean[keep].copy()
clean.columns = clean.columns.strftime("%Y-%m-%d")
clean.insert(0, "CONS_NO", ids[keep].values)
clean["MISSING_PCT"] = missing_per_customer[keep].round(2).values
clean["FLAG"] = labels[keep].values

clean.to_csv("../data/clean.csv", index=False)
print("Saved! Shape:", clean.shape)
print("Theft rate now:", round(clean["FLAG"].mean() * 100, 2), "%")

Customers removed: 83
Saved! Shape: (42289, 1037)
Theft rate now: 8.5 %
